In [0]:
from pyspark.sql.functions import to_date, col, when, year, month, dayofmonth, dayofweek, date_format
from datetime import datetime
import requests
import csv
import time
import json

In [0]:
# Authenticating Azure Data Lake Storage Gen2
storage_account = "capitaledgestorageacct"

# get storage access key
storage_key = "azO2UKU4KCiR7J1rvy3QhGcu6WbOGDc1rqXn4aFC0HA5J+DODfKL/dWAibXVhKjyg6NqBYlGcj99+AStE3iwkg=="

# configure databricks to access storage account
spark.conf.set(
  f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
  storage_key
)

BRONZE = f"abfss://bronze-capitaledge@{storage_account}.dfs.core.windows.net/"
SILVER = f"abfss://silver-capitaledge@{storage_account}.dfs.core.windows.net/"
DATA_TRACKING = f"abfss://capitaledge-data-tracking@{storage_account}.dfs.core.windows.net/"


In [0]:

df = spark.read.parquet(f'{BRONZE}bronze_capitaledge/').orderBy(col("date").desc())
display(df)

In [0]:
# Getting the last stock date from latest stock date ADLS
last_date_df = spark.read.option("multiline", "true").json(f'{DATA_TRACKING}/silver_last_date.json')

# Getting the actual last date value
last_date = last_date_df.where(col("1-pipeline") == "silver").select("2-latest_date").first()[0]

# Getting the actual ealier date value
#### early_date = last_date_df.where(col("1-pipeline") == "silver").select("3-earlier_date").first()[0]

# Converting to date type
last_date = datetime.strptime(last_date, "%Y-%m-%d").date()
#### early_date = datetime.strptime(early_date, "%Y-%m-%d").date()

print(last_date)
#### print(early_date)

In [0]:
# Filtering only rows greater than the last date in our watermark

df_filtered = df.filter(col("date") > last_date) 
display(df_filtered)

In [0]:
df_filtered = df_filtered \
    .withColumnRenamed("1. open", "open") \
    .withColumnRenamed("2. high", "high") \
    .withColumnRenamed("3. low", "low") \
    .withColumnRenamed("4. close", "close") \
    .withColumnRenamed("5. volume", "volume")


In [0]:
# Casting the appropriate datatypes for the columns
df_transformed = df_filtered \
    .withColumn("open", col("open").cast("decimal(10,4)")) \
    .withColumn("high", col("high").cast("decimal(10,4)")) \
    .withColumn("low", col("low").cast("decimal(10,4)")) \
    .withColumn("close", col("close").cast("decimal(10,4)")) \
    .withColumn("volume", col("volume").cast("long")) \
    .withColumn("date", to_date("date", "yyyy-MM-dd"))

display(df_transformed)

In [0]:
# Getting the latest stock date to update our watermark in ADLS
new_last_date = df_transformed.orderBy(col("date").desc()).select("date").first()[0]

# Updating and overwriting the latest stock date to ADLS
silver_last_date = {
    "1-pipeline": "silver",
    "2-latest_date": new_last_date,
    "3-earlier_date": last_date
}

file_path = f"{DATA_TRACKING}/silver_last_date.json"

# Save the JSON data
json_data = json.dumps(silver_last_date, indent=4, default=str)
dbutils.fs.put(file_path, json_data, overwrite=True)

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-8262218165300993>, line 2
      1 # Getting the latest stock date to update our watermark in ADLS
----> 2 new_last_date = df_transformed.orderBy(col("date").desc()).select("date").first()[0]
      4 # Updating and overwriting the latest stock date to ADLS
      5 silver_last_date = {
      6     "1-pipeline": "silver",
      7     "2-latest_date": new_last_date,
      8     "3-earlier_date": last_date
      9 }

TypeError: 'NoneType' object is not subscriptable

In [0]:
# Save the transformed DataFrame to the Silver container
silver_output_path = f"{SILVER}silver_capitaledge/"

# Append DataFrame to Silver container in Parquet format
df_transformed.write.mode('append').parquet(silver_output_path)